# 04_02 — Limpieza y transformación de datos meteorológicos NOAA

**Proyecto Big Data · Fase 04 · PySpark / Parquet · Año de estudio: 2024**

## Objetivo

Convertir las observaciones meteorológicas obtenidas de **NOAA Climate Data Online (GHCND)** para la estación **NY CITY CENTRAL PARK** (`GHCND:USW00094728`) en una tabla **diaria, validada y reutilizable** para la integración con NYC Taxi.

**Contrato de entrada → salida**

| Capa | Granularidad | Volumen observado | Ubicación |
|:--|:--|--:|:--|
| Raw (JSON) | Una observación por fecha y variable (`long`) | 1,826 registros en 2 páginas | `data/raw/weather/year=2024/station=USW00094728/` |
| Silver (Parquet) | Una fila por día (`wide`) | 366 fechas únicas | `data/processed/weather/silver/` |

**Criterios de diseño:** no modificar los JSON Raw; comprobar unicidad antes del `pivot`; mantener los días sin datos mediante un calendario completo; conservar los faltantes como `NULL` y documentar las reglas de calidad.

> **Dependencias:** el módulo `src/etl/weather_cleaning.py` (función `transform_weather_daily`) debe estar disponible en el repositorio. Los archivos Raw y las salidas de este notebook corresponden al conjunto NOAA descargado en la fase 03.

## 1. Preparación del entorno

Se identifica la raíz del repositorio y se inicializa una sesión local de Spark. La búsqueda basada en las carpetas `data/` y `notebooks/` permite ejecutar el notebook desde distintas ubicaciones del proyecto. `local[4]` limita a cuatro hilos de ejecución la sesión local.

**Nota de reproducibilidad:** las rutas y resultados mostrados en esta versión provienen de una ejecución en Windows; las pruebas automatizadas se ejecutaron satisfactoriamente en Docker. Para reproducir el flujo, conviene seleccionar un único entorno por sesión y mantener la ruta del proyecto montada de forma coherente.

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# --------------------------------------------------
# 1. Localizar la raíz del proyecto
# --------------------------------------------------

def find_project_root(start: Path) -> Path:
    """Localiza la raíz del repositorio de Big Data."""

    for candidate in [start, *start.parents]:

        if (
            (candidate / "data").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

# --------------------------------------------------
# 2. Inicializar Spark
# --------------------------------------------------

spark = (
    SparkSession.builder
    .appName("04_02_noaa_data_cleaning")
    .master("local[4]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Proyecto:", PROJECT_ROOT)
print("Versión Spark:", spark.version)

Proyecto: c:\Users\derec\Desktop\Proyectos\BigData_project
Versión Spark: 4.1.3


## 2. Ingesta de los archivos originales

La API de NOAA devolvió dos respuestas JSON paginadas. Cada archivo contiene metadatos de consulta y un arreglo `results` con las observaciones. Con `multiLine=True`, Spark lee cada respuesta como un documento JSON incluso cuando abarca varias líneas.

En este punto **no se modifica el contenido meteorológico**: únicamente se inspeccionan el número de archivos y el esquema de lectura. La validación posterior usa el número esperado de observaciones para detectar páginas omitidas o duplicadas.

In [2]:
# --------------------------------------------------
# 1. Localizar archivos JSON originales
# --------------------------------------------------

NOAA_RAW_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "weather"
    / "year=2024"
    / "station=USW00094728"
)

noaa_files = sorted(
    NOAA_RAW_PATH.glob("page_*.json")
)

print("Archivos encontrados:", len(noaa_files))

for path in noaa_files:
    print(path.relative_to(PROJECT_ROOT))

assert len(noaa_files) == 2, (
    "Se esperaban 2 páginas JSON de NOAA."
)

# --------------------------------------------------
# 2. Leer respuestas originales de la API
# --------------------------------------------------

df_noaa_raw = (
    spark.read
    .option("multiLine", "true")
    .json([str(path) for path in noaa_files])
)

# Revisar estructura original
df_noaa_raw.printSchema()

Archivos encontrados: 2
data\raw\weather\year=2024\station=USW00094728\page_001.json
data\raw\weather\year=2024\station=USW00094728\page_002.json
root
 |-- metadata: struct (nullable = true)
 |    |-- resultset: struct (nullable = true)
 |    |    |-- count: long (nullable = true)
 |    |    |-- limit: long (nullable = true)
 |    |    |-- offset: long (nullable = true)
 |-- results: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attributes: string (nullable = true)
 |    |    |-- datatype: string (nullable = true)
 |    |    |-- date: string (nullable = true)
 |    |    |-- station: string (nullable = true)
 |    |    |-- value: double (nullable = true)



### 2.1. Resultado de la lectura

Se reconocieron los dos archivos (`page_001.json` y `page_002.json`) y el esquema contiene `metadata.resultset` y `results` como arreglo de estructuras. Dentro de cada observación están `station`, `date`, `datatype`, `value` y `attributes`.

La información de `attributes` se **conserva durante la normalización** para asegurar trazabilidad, pero no se interpreta como una validación de calidad meteorológica adicional en esta fase.

## 3. Normalización del JSON (*long format*)

La columna `results` contiene múltiples observaciones dentro de una sola fila. `explode(results)` convierte cada elemento de ese arreglo en una fila individual. A continuación, `select` proyecta las variables necesarias y asigna tipos explícitos.

La fecha se obtiene de los primeros diez caracteres de `observation.date` y se transforma a tipo `date` (sin hora). El resultado tiene el esquema lógico:

`(station, weather_date, datatype, value, attributes)`.

**Unidades esperadas del proceso de descarga:** `TMAX`/`TMIN` en °C; `PRCP` y `SNOW` en mm; `AWND` en m/s. Estas unidades dependen de haber solicitado la API con `units=metric` en la fase 03: **el JSON mostrado no almacena por sí mismo esa opción**, por lo que debe conservarse documentada en la ingesta. No se aplican conversiones adicionales aquí.

In [3]:
# --------------------------------------------------
# 1. Extraer los registros de results
# --------------------------------------------------

df_noaa_exploded = (
    df_noaa_raw
    .select(
        F.explode("results").alias("observation")
    )
)

# --------------------------------------------------
# 2. Construir esquema normalizado
# --------------------------------------------------

df_weather_long = (
    df_noaa_exploded
    .select(

        # Identificador de la estación
        F.col("observation.station")
        .cast("string")
        .alias("station"),

        # Fecha meteorológica sin componente horario
        F.to_date(
            F.substring(
                F.col("observation.date"),
                1,
                10
            )
        ).alias("weather_date"),

        # Tipo de observación
        F.col("observation.datatype")
        .cast("string")
        .alias("datatype"),

        # Valor numérico en unidades métricas
        F.col("observation.value")
        .cast("double")
        .alias("value"),

        # Atributos originales de NOAA
        F.col("observation.attributes")
        .alias("attributes")
    )
)

# --------------------------------------------------
# 3. Inspeccionar resultados
# --------------------------------------------------

df_weather_long.printSchema()

df_weather_long.orderBy(
    "weather_date", "datatype"
).show(15, truncate=False)

root
 |-- station: string (nullable = true)
 |-- weather_date: date (nullable = true)
 |-- datatype: string (nullable = true)
 |-- value: double (nullable = true)
 |-- attributes: string (nullable = true)

+-----------------+------------+--------+-----+----------+
|station          |weather_date|datatype|value|attributes|
+-----------------+------------+--------+-----+----------+
|GHCND:USW00094728|2024-01-01  |AWND    |1.5  |,,W,      |
|GHCND:USW00094728|2024-01-01  |PRCP    |0.8  |,,W,2400  |
|GHCND:USW00094728|2024-01-01  |SNOW    |0.0  |,,W,      |
|GHCND:USW00094728|2024-01-01  |TMAX    |8.3  |,,W,2400  |
|GHCND:USW00094728|2024-01-01  |TMIN    |1.7  |,,W,2400  |
|GHCND:USW00094728|2024-01-02  |AWND    |1.8  |,,W,      |
|GHCND:USW00094728|2024-01-02  |PRCP    |0.0  |,,W,2400  |
|GHCND:USW00094728|2024-01-02  |SNOW    |0.0  |,,W,      |
|GHCND:USW00094728|2024-01-02  |TMAX    |5.6  |,,W,2400  |
|GHCND:USW00094728|2024-01-02  |TMIN    |-1.6 |,,W,2400  |
|GHCND:USW00094728|2024-01-

El resultado tiene cinco columnas con los tipos esperados. Como ejemplo, el **1 de enero de 2024** aparece desglosado en cinco observaciones (AWND, PRCP, SNOW, TMAX y TMIN) para una misma estación. Eso confirma la granularidad original **estación–fecha–tipo de medición**.

## 4. Validaciones previas a la transformación

Antes del `pivot` verificamos cuatro contratos de calidad:

1. **Integridad de ingesta:** 1,826 observaciones y una sola estación.
2. **Cobertura temporal:** 366 fechas diferentes en 2024 (año bisiesto).
3. **Cobertura por variable:** TMAX, TMIN, PRCP y SNOW cuentan con 366 observaciones; AWND tiene 362.
4. **Unicidad de la clave** `(station, weather_date, datatype)`: no deben existir duplicados.

El punto 4 es fundamental: la función modular utiliza `first(value)` para pivotar. Si existieran duplicados con valores diferentes, dicha agregación elegiría uno sin resolver el conflicto de origen. Por eso **no deben aplicarse deduplicaciones automáticas** sin analizarlas primero.

In [4]:
# --------------------------------------------------
# 1. Validar volumen y cobertura temporal
# --------------------------------------------------

total_records = df_weather_long.count()

total_dates = (
    df_weather_long
    .select("weather_date")
    .distinct()
    .count()
)

total_stations = (
    df_weather_long
    .select("station")
    .distinct()
    .count()
)

print("Observaciones:", total_records)
print("Fechas diferentes:", total_dates)
print("Estaciones:", total_stations)

assert total_records == 1826
assert total_dates == 366
assert total_stations == 1

# --------------------------------------------------
# 2. Cobertura por variable meteorológica
# --------------------------------------------------

df_coverage = (
    df_weather_long
    .groupBy("datatype")
    .agg(
        F.count("*").alias("observations"),
        F.countDistinct("weather_date").alias(
            "available_dates"
        ),
        F.sum(
            F.col("value").isNull().cast("long")
        ).alias("null_values")
    )
    .orderBy("datatype")
)

df_coverage.show(truncate=False)

# --------------------------------------------------
# 3. Detectar observaciones duplicadas
# --------------------------------------------------

df_duplicates = (
    df_weather_long
    .groupBy(
        "station",
        "weather_date",
        "datatype"
    )
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = df_duplicates.count()

print("Claves duplicadas:", duplicate_count)

assert duplicate_count == 0

Observaciones: 1826
Fechas diferentes: 366
Estaciones: 1
+--------+------------+---------------+-----------+
|datatype|observations|available_dates|null_values|
+--------+------------+---------------+-----------+
|AWND    |362         |362            |0          |
|PRCP    |366         |366            |0          |
|SNOW    |366         |366            |0          |
|TMAX    |366         |366            |0          |
|TMIN    |366         |366            |0          |
+--------+------------+---------------+-----------+

Claves duplicadas: 0


### 4.1. Interpretación de la calidad Raw

La ejecución confirma **1,826 observaciones, 366 fechas distintas, una estación y cero claves duplicadas**. Las cinco variables no muestran `value = NULL` dentro de los registros descargados. La diferencia está en la **ausencia de cuatro registros AWND** (no en valores nulos dentro de registros AWND).

| Variable | Registros | Cobertura de días | Interpretación |
|:--|--:|--:|:--|
| AWND | 362 | 98.91% | Faltan cuatro mediciones diarias de viento |
| PRCP | 366 | 100% | Cobertura completa de precipitación |
| SNOW | 366 | 100% | Cobertura completa de nieve |
| TMAX | 366 | 100% | Cobertura completa de máxima diaria |
| TMIN | 366 | 100% | Cobertura completa de mínima diaria |

En consecuencia, los nulos de AWND **aparecerán después de aplicar `pivot` y unir el calendario**, lo que permite mantenerlos visibles sin imputar valores ficticios.

In [5]:
# Validación adicional del contrato de ingesta, antes del pivot.
# El tamaño es pequeño (cinco tipos): recopilar esta lista es seguro.
EXPECTED_COUNTS = {
    "AWND": 362,
    "PRCP": 366,
    "SNOW": 366,
    "TMAX": 366,
    "TMIN": 366,
}

observed_counts = {
    row["datatype"]: row["observations"]
    for row in df_coverage.collect()
}

assert observed_counts == EXPECTED_COUNTS, (
    f"Cobertura diferente de la esperada: {observed_counts}"
)

# La estación, el año y las fechas deben coincidir con el contrato.
invalid_source_rows = df_weather_long.filter(
    (~F.col("station").eqNullSafe("GHCND:USW00094728"))
    | F.col("weather_date").isNull()
    | (F.year("weather_date") != 2024)
    | F.col("value").isNull()
).count()

assert invalid_source_rows == 0
print("Contrato Raw validado: tipos, volúmenes, estación y fechas.")

Contrato Raw validado: tipos, volúmenes, estación y fechas.


## 5. Transformación a una tabla meteorológica diaria

Se reutiliza `transform_weather_daily` del módulo `src/etl/weather_cleaning.py`. Su lógica queda separada del notebook para poder probarla y ejecutarla también desde scripts o procesos ETL.

**Pasos implementados:**

1. Seleccionar la estación y las variables AWND, PRCP, SNOW, TMAX y TMIN para 2024.
2. Agrupar por `weather_date` y realizar un `pivot` para pasar de formato largo a ancho.
3. Crear el **calendario completo de 366 fechas** y usar un `left join` para preservar días aunque falten observaciones.
4. Renombrar las variables para hacer explícitas sus unidades (`awnd_mps`, `prcp_mm`, `snow_mm`, `tmax_c`, `tmin_c`) e incorporar indicadores útiles para análisis posteriores.

### Variables derivadas

El punto medio de temperaturas extremas y el rango térmico son:

$$
T_{\text{mid},d}=\frac{T_{\max,d}+T_{\min,d}}{2},
\qquad
R_{T,d}=T_{\max,d}-T_{\min,d}.
$$

`temperature_midpoint_c` es una **aproximación basada en extremos**, no la temperatura media diaria calculada a partir de mediciones horarias. `has_precipitation` y `has_snow` indican cantidades estrictamente mayores a cero; si falta la medición, su resultado permanece `NULL`.

Las banderas de calidad permiten reconocer viento faltante, máxima inferior a mínima, precipitación negativa y nieve negativa. Las banderas no eliminan observaciones.

In [6]:
import sys
import importlib

# Permitir importar módulos desde la raíz del proyecto
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.etl.weather_cleaning as weather_cleaning

importlib.reload(weather_cleaning)

# Aplicar transformaciones
df_weather_daily = (
    weather_cleaning.transform_weather_daily(
        df=df_weather_long,
        spark=spark,
        year=2024
    )
)

# Revisar estructura y primeras observaciones
df_weather_daily.printSchema()

df_weather_daily.show(10, truncate=False)

root
 |-- weather_date: date (nullable = false)
 |-- awnd_mps: double (nullable = true)
 |-- prcp_mm: double (nullable = true)
 |-- snow_mm: double (nullable = true)
 |-- tmax_c: double (nullable = true)
 |-- tmin_c: double (nullable = true)
 |-- station: string (nullable = false)
 |-- temperature_midpoint_c: double (nullable = true)
 |-- temperature_range_c: double (nullable = true)
 |-- has_precipitation: boolean (nullable = true)
 |-- has_snow: boolean (nullable = true)
 |-- flag_missing_awnd: boolean (nullable = false)
 |-- flag_invalid_temperature: boolean (nullable = false)
 |-- flag_negative_precipitation: boolean (nullable = false)
 |-- flag_negative_snow: boolean (nullable = false)

+------------+--------+-------+-------+------+------+-----------------+----------------------+-------------------+-----------------+--------+-----------------+------------------------+---------------------------+------------------+
|weather_date|awnd_mps|prcp_mm|snow_mm|tmax_c|tmin_c|station       

### 5.1. Interpretación de las primeras filas

El DataFrame transformado tiene **una fila por fecha** y quince columnas (cinco mediciones principales, identificadores, derivados e indicadores). El primer día conserva los valores originales: TMAX = 8.3 °C, TMIN = 1.7 °C y PRCP = 0.8 mm; de aquí resulta un punto medio de **5.0 °C**.

Las representaciones como `6.6000000000000005` se explican por la precisión numérica de `double`; no indican un error meteorológico. Para presentar tablas o gráficas puede redondearse visualmente, sin alterar la precisión almacenada.

## 6. Validación de la tabla transformada

La capa diaria debe conservar **366 fechas distintas** y mostrar explícitamente las cuatro observaciones faltantes de `AWND`. Además, no debe presentar temperaturas máximas inferiores a las mínimas ni cantidades negativas de lluvia o nieve, de acuerdo con las reglas básicas de consistencia física implementadas.

Los conteos de días con precipitación y nieve se basan en mediciones positivas (`PRCP > 0` y `SNOW > 0`). Son conteos de días meteorológicos, no de eventos ni de horas con lluvia o nieve.

In [7]:
# --------------------------------------------------
# 1. Validación estructural
# --------------------------------------------------

total_dates = df_weather_daily.count()

unique_dates = (
    df_weather_daily
    .select("weather_date")
    .distinct()
    .count()
)

assert total_dates == 366
assert unique_dates == 366

# --------------------------------------------------
# 2. Diagnóstico de calidad
# --------------------------------------------------

df_validation = df_weather_daily.agg(

    F.count("*").alias("total_days"),

    F.sum(
        F.col("flag_missing_awnd").cast("long")
    ).alias("missing_awnd"),

    F.sum(
        F.col("flag_invalid_temperature").cast("long")
    ).alias("invalid_temperature"),

    F.sum(
        F.col("flag_negative_precipitation").cast("long")
    ).alias("negative_precipitation"),

    F.sum(
        F.col("flag_negative_snow").cast("long")
    ).alias("negative_snow"),

    F.sum(
        F.col("has_precipitation").cast("long")
    ).alias("days_with_precipitation"),

    F.sum(
        F.col("has_snow").cast("long")
    ).alias("days_with_snow")
)

df_validation.show(truncate=False)

# --------------------------------------------------
# 3. Revisar las fechas con viento faltante
# --------------------------------------------------

(
    df_weather_daily
    .filter(F.col("flag_missing_awnd"))
    .select("weather_date", "awnd_mps")
    .orderBy("weather_date")
    .show(truncate=False)
)

# --------------------------------------------------
# 4. Comprobar cobertura esperada
# --------------------------------------------------

assert (
    df_weather_daily
    .filter(F.col("awnd_mps").isNull())
    .count()
) == 4

print("Validaciones estructurales correctas.")

+----------+------------+-------------------+----------------------+-------------+-----------------------+--------------+
|total_days|missing_awnd|invalid_temperature|negative_precipitation|negative_snow|days_with_precipitation|days_with_snow|
+----------+------------+-------------------+----------------------+-------------+-----------------------+--------------+
|366       |4           |0                  |0                     |0            |125                    |8             |
+----------+------------+-------------------+----------------------+-------------+-----------------------+--------------+

+------------+--------+
|weather_date|awnd_mps|
+------------+--------+
|2024-01-24  |NULL    |
|2024-01-27  |NULL    |
|2024-06-23  |NULL    |
|2024-06-24  |NULL    |
+------------+--------+

Validaciones estructurales correctas.


### 6.1. Resultados y decisión sobre valores faltantes

El dataset presenta **366 días, cuatro nulos de viento, cero anomalías físicas básicas**, **125 días con precipitación** (34.15%) y **8 días con nieve** (2.19%). Las fechas sin AWND son:

- 2024-01-24 y 2024-01-27.
- 2024-06-23 y 2024-06-24.

**Decisión:** conservar `awnd_mps = NULL` y `flag_missing_awnd = true` en esas fechas. Imputar cero significaría registrar ausencia de viento sin evidencia. La imputación, si se necesita posteriormente para un modelo, deberá justificarse y evaluarse por separado.

Las mediciones se atribuyen a **una sola estación (Central Park)**: representan esa estación y no necesariamente las condiciones exactas de todos los barrios de Nueva York.

In [8]:
# Verificación complementaria: el resto de mediciones diarias
# debería estar completo en este corte concreto de NOAA 2024.
required_complete = ["prcp_mm", "snow_mm", "tmax_c", "tmin_c"]
missing_other_measurements = df_weather_daily.filter(
    F.expr(" OR ".join(f"{name} IS NULL" for name in required_complete))
).count()

assert missing_other_measurements == 0
assert df_weather_daily.filter(F.col("weather_date").isNull()).count() == 0

print("Cobertura Silver: faltantes limitados a cuatro días de AWND.")

Cobertura Silver: faltantes limitados a cuatro días de AWND.


## 7. Pruebas automatizadas fuera del notebook

La función `transform_weather_daily` se verificó con datos artificiales en `tests/test_weather_cleaning.py`. La prueba cubre: calendario bisiesto completo, transformación `pivot`, variables derivadas, conservación de días sin mediciones y detección de anomalías.

**Comando desde la raíz del proyecto (Docker):**

```bash
docker compose run --rm spark python -m pytest tests/test_weather_cleaning.py -v -s
```

**Resultado comunicado de la ejecución:** `1 passed` en Linux, Python 3.11.17. Una ejecución previa de la prueba en Windows había fallado al inicializar Py4J/Spark; esa incidencia es del entorno de pruebas y no invalida la persistencia y las validaciones del dataset, que terminaron correctamente. Para nuevas ejecuciones automatizadas se recomienda usar el entorno Docker verificado.

> Nota: el archivo de prueba forma parte del repositorio, no de este `.ipynb`. Aquí se documenta su ejecución, sin simular que la prueba se ejecutó desde el propio notebook.

## 8. Persistencia en la capa Silver (Parquet)

Se guarda la tabla diaria en `data/processed/weather/silver/` con compresión **Snappy**. Por tratarse de solo 366 filas, `coalesce(1)` es una decisión razonable para generar un único archivo de datos Parquet (junto con archivos de metadatos de Spark). No utilizamos particionamiento mensual ni escritura incremental porque producirían complejidad y fragmentación innecesarias para este volumen.

`mode("overwrite")` reemplaza el contenido anterior de **esa ruta Silver**; los archivos JSON Raw no se modifican. Si la ruta contiene datos que deben conservarse, debe realizarse una copia antes de sobrescribirla.

In [9]:
# --------------------------------------------------
# 1. Definir ruta de almacenamiento
# --------------------------------------------------

WEATHER_SILVER_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "weather"
    / "silver"
)

WEATHER_SILVER_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

# --------------------------------------------------
# 2. Guardar dataset procesado
# --------------------------------------------------

(
    df_weather_daily
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("compression", "snappy")
    .parquet(str(WEATHER_SILVER_PATH))
)

print("Dataset meteorológico almacenado.")
print("Ruta:", WEATHER_SILVER_PATH)

Dataset meteorológico almacenado.
Ruta: c:\Users\derec\Desktop\Proyectos\BigData_project\data\processed\weather\silver


## 9. Comprobación de lectura y persistencia

Se vuelve a leer **desde el Parquet almacenado**, para evitar depender únicamente del DataFrame en memoria. La validación comprueba registros y fechas únicas, límites del calendario, nulos de AWND y conteos de días con lluvia y nieve. Esta comprobación verifica que los valores esperados sobrevivieron al ciclo de **escritura → lectura**.

In [10]:
from datetime import date

# --------------------------------------------------
# 1. Leer el dataset almacenado
# --------------------------------------------------

df_weather_saved = spark.read.parquet(
    str(WEATHER_SILVER_PATH)
)

# --------------------------------------------------
# 2. Comprobar integridad
# --------------------------------------------------

validation = df_weather_saved.agg(

    F.count("*").alias("total_days"),

    F.countDistinct("weather_date").alias(
        "unique_dates"
    ),

    F.min("weather_date").alias("start_date"),

    F.max("weather_date").alias("end_date"),

    F.sum(
        F.col("flag_missing_awnd").cast("long")
    ).alias("missing_awnd"),

    F.sum(
        F.col("has_precipitation").cast("long")
    ).alias("precipitation_days"),

    F.sum(
        F.col("has_snow").cast("long")
    ).alias("snow_days")

).first()

# --------------------------------------------------
# 3. Validar resultados esperados
# --------------------------------------------------

assert validation["total_days"] == 366
assert validation["unique_dates"] == 366

assert validation["start_date"] == date(2024, 1, 1)
assert validation["end_date"] == date(2024, 12, 31)

assert validation["missing_awnd"] == 4
assert validation["precipitation_days"] == 125
assert validation["snow_days"] == 8

print("Todas las validaciones de Silver pasaron.")
print("Registros almacenados:", validation["total_days"])

Todas las validaciones de Silver pasaron.
Registros almacenados: 366


## 10. Conclusiones y preparación para la integración

**Resultados obtenidos:**

- Se normalizaron **1,826 observaciones NOAA** obtenidas de dos archivos JSON y se transformaron en una tabla **Silver de 366 días de 2024**.
- La granularidad final es **una fila por `weather_date`** para la estación `GHCND:USW00094728`.
- Se conservó toda la cobertura de fechas y se identificaron **cuatro mediciones faltantes de AWND**, sin imputaciones arbitrarias.
- Las comprobaciones estructurales y físicas básicas pasaron, y el conjunto se almacenó y leyó correctamente en Parquet.
- El módulo reutilizable está cubierto por una prueba de `pytest` que **pasó en Docker**, según la ejecución registrada durante el desarrollo.

**Limitaciones y precauciones:**

1. La tabla corresponde a **una estación** y a **resolución diaria**; no permite conocer el clima exacto durante un viaje ni en todas las zonas de NYC.
2. Las banderas de consistencia **no sustituyen** una evaluación completa de la calidad de NOAA (`attributes` y otras banderas no interpretadas en esta fase).
3. El punto medio de TMAX/TMIN no es una media horaria observada.
4. En un uso predictivo, **no se debe utilizar la precipitación total del día como una variable conocida anticipadamente** para predecir viajes ocurridos antes de terminar ese día.

**Siguiente fase — `04_03`**: integrar con la capa Silver de NYC Taxi mediante `taxi.pickup_date = weather.weather_date`, como una unión *many-to-one*. Antes del `join` se verificará de nuevo la unicidad de `weather_date` y después se contrastarán los conteos de viajes para detectar multiplicaciones o pérdidas de registros. También se revisará la compatibilidad de la fecha local del viaje con la convención diaria de la estación meteorológica.